<a href="https://colab.research.google.com/github/KiaNoForte/Cryptography/blob/main/CryptoLab04B_Stream_Ciphers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crypto Lab 04B - Stream Ciphers

**Course:** CYBR 3570 - Applied Cryptography  
**Theme:** From one-time pads to modern keystream generators

## Today's Big Question

**What goes wrong when a stream cipher repeats its keystream?**

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain how stream ciphers encrypt and decrypt using XOR.
2. Distinguish a key, nonce, counter, and keystream.
3. Demonstrate why reusing a key/nonce pair is dangerous.
4. Implement a toy feedback shift register.
5. Explain why linear feedback shift registers are not secure by themselves.
6. Use a modern library stream-cipher-style construction appropriately.
7. Connect RC4, WEP, Salsa20, and ChaCha20 to real-world cryptographic engineering.

## 1. Stream Ciphers in One Sentence

A stream cipher generates a pseudorandom **keystream** from a secret key and public nonce, then XORs that keystream with plaintext to produce ciphertext.

Encryption:

`C = P XOR KS`

Decryption:

`P = C XOR KS`

The same operation works both ways because XOR cancels itself.

In [1]:
# XOR refresher
print(0 ^ 0)
print(0 ^ 1)
print(1 ^ 0)
print(1 ^ 1)

# XOR cancels itself:
value = 0b10101100
mask = 0b11001010
cipher = value ^ mask
recovered = cipher ^ mask

print(bin(value), bin(cipher), bin(recovered))
assert recovered == value

0
1
1
0
0b10101100 0b1100110 0b10101100


## 2. XORing Byte Strings

Python's `^` operator works on integers, not directly on byte strings. We will define a helper that XORs two byte strings of the same length.

In [2]:
def xor_bytes(a: bytes, b: bytes) -> bytes:
    """Return the bytewise XOR of two equal-length byte strings."""
    if len(a) != len(b):
        raise ValueError("Inputs must have the same length")
    return bytes(x ^ y for x, y in zip(a, b))

plaintext = b"STREAM CIPHERS!"
keystream = bytes([0x55] * len(plaintext))
ciphertext = xor_bytes(plaintext, keystream)
recovered = xor_bytes(ciphertext, keystream)

print("plaintext :", plaintext)
print("keystream :", keystream.hex())
print("ciphertext:", ciphertext.hex())
print("recovered :", recovered)
assert recovered == plaintext

plaintext : b'STREAM CIPHERS!'
keystream : 555555555555555555555555555555
ciphertext: 06010710141875161c051d10070674
recovered : b'STREAM CIPHERS!'


## 3. A Toy Stream Cipher

The following function is **not secure**. It is intentionally simple so we can see how stream cipher encryption works.

It expands a key and nonce into a keystream by hashing the key, nonce, and counter repeatedly.

This is an educational construction only. Do not use it for real encryption.

In [3]:
import hashlib
import secrets


def toy_keystream(key: bytes, nonce: bytes, length: int) -> bytes:
    """Generate a toy keystream using SHA-256(key || nonce || counter)."""
    output = bytearray()
    counter = 0
    while len(output) < length:
        block = hashlib.sha256(
            key + nonce + counter.to_bytes(8, "big")
        ).digest()
        output.extend(block)
        counter += 1
    return bytes(output[:length])


def toy_stream_encrypt(key: bytes, nonce: bytes, plaintext: bytes) -> bytes:
    ks = toy_keystream(key, nonce, len(plaintext))
    return xor_bytes(plaintext, ks)

key = secrets.token_bytes(32)
nonce = secrets.token_bytes(12)
message = b"Stream ciphers turn keys into keystreams."

ct = toy_stream_encrypt(key, nonce, message)
pt = toy_stream_encrypt(key, nonce, ct)

print("nonce:", nonce.hex())
print("ct   :", ct.hex())
print("pt   :", pt)
assert pt == message

nonce: 4c1bf29a3bc710135dcd6e7a
ct   : 2c3c789c48bd90c3e6da879fe7bb7563662dc739a0c5435211a64bce149b4b53c5b4d97f28d5d4b030
pt   : b'Stream ciphers turn keys into keystreams.'


## 4. The Key/Nonce Rule

A stream cipher may safely reuse the same **key** with different **nonces**.

A stream cipher may safely reuse the same **nonce** with different **keys**.

A stream cipher must **not** reuse the same key and nonce together.

When the same key and nonce are reused, the same keystream is reused. That creates a two-time pad problem.

In [4]:
key = secrets.token_bytes(32)
nonce = secrets.token_bytes(12)

p1 = b"Attack at dawn. Send ten units."
p2 = b"Retreat tonight. Send two men."

# Make equal lengths for a clean demonstration.
min_len = min(len(p1), len(p2))
p1 = p1[:min_len]
p2 = p2[:min_len]

c1 = toy_stream_encrypt(key, nonce, p1)
c2 = toy_stream_encrypt(key, nonce, p2)

print("c1 XOR c2:", xor_bytes(c1, c2))
print("p1 XOR p2:", xor_bytes(p1, p2))
assert xor_bytes(c1, c2) == xor_bytes(p1, p2)

c1 XOR c2: b'\x13\x11\x00\x13\x06\nTA\x00O\n\x08\x10\x06Z\x0es6\x0b\nDT\x11\x19OU\x03\x0c\x1a]'
p1 XOR p2: b'\x13\x11\x00\x13\x06\nTA\x00O\n\x08\x10\x06Z\x0es6\x0b\nDT\x11\x19OU\x03\x0c\x1a]'


## 5. Known-Plaintext Recovery After Keystream Reuse

If an attacker knows or guesses one plaintext, the reused keystream lets them recover the other plaintext.

Given:

`C1 = P1 XOR KS`

Then:

`KS = C1 XOR P1`

And:

`P2 = C2 XOR KS`

In [5]:
known_p1 = p1
recovered_ks = xor_bytes(c1, known_p1)
recovered_p2 = xor_bytes(c2, recovered_ks)

print("known p1    :", known_p1)
print("recovered p2:", recovered_p2)
print("actual p2   :", p2)
assert recovered_p2 == p2

known p1    : b'Attack at dawn. Send ten units'
recovered p2: b'Retreat tonight. Send two men.'
actual p2   : b'Retreat tonight. Send two men.'


## 6. Exercise: Explain the Failure

In your own words, answer:

1. Why does reusing a key/nonce pair repeat the keystream?
2. Why does XORing two ciphertexts remove the keystream?
3. Why is this similar to reusing a one-time pad?

In [6]:
# Write your answer in this Markdown cell or as comments here.

Stream ciphers generate their pseudorandom keystream deterministically from the combination of a secret key and a nonce. Because the generation process is fully deterministic, feeding in the exact same key and nonce inputs will always produce the exact same keystream output.

Because XOR cancellation properties dictate that any value XORed with itself equals zero and any value XORed with zero remains unchanged. When you XOR two ciphertexts generated with the same keystream. The keystream cancels out completely, leaving only the XOR sum of the two original plaintexts.

A one-time pad (OTP) achieves perfect secrecy only under the strict condition that the key material is never reused. Reusing a stream cipher keystream creates a classic "two-time pad" vulnerability—it eliminates the random obfuscation protecting the underlying data, exposing structural patterns, language redundancies, or known plaintext snippets that allow an attacker to reconstruct the original messages.

## 7. Feedback Shift Registers

Hardware-oriented stream ciphers often use feedback shift registers (FSRs).

A simple FSR:

1. Stores a register of bits.
2. Outputs one bit at each step.
3. Shifts the register.
4. Computes a new bit using a feedback function.

We will implement a toy 4-bit FSR.

In [7]:
def fsr_step(state: tuple[int, ...]) -> tuple[int, tuple[int, ...]]:
    """One step of a toy 4-bit FSR where feedback is XOR of all bits."""
    if len(state) != 4:
        raise ValueError("This toy FSR expects exactly 4 bits")
    output = state[0]
    feedback = state[0] ^ state[1] ^ state[2] ^ state[3]
    next_state = state[1:] + (feedback,)
    return output, next_state

state = (1, 1, 0, 0)
outputs = []
states = [state]
for _ in range(12):
    bit, state = fsr_step(state)
    outputs.append(bit)
    states.append(state)

print("outputs:", ''.join(str(b) for b in outputs))
print("states:")
for s in states:
    print(''.join(str(b) for b in s))

outputs: 110001100011
states:
1100
1000
0001
0011
0110
1100
1000
0001
0011
0110
1100
1000
0001


## 8. Period of a Toy FSR

The period is the number of updates before the state repeats.

Short periods are bad because they create repeating keystream patterns.

In [8]:
def fsr_period(initial_state: tuple[int, ...], step_function) -> int:
    seen = {}
    state = initial_state
    t = 0
    while state not in seen:
        seen[state] = t
        _, state = step_function(state)
        t += 1
    return t - seen[state]

for init in [(1,1,0,0), (0,0,0,0), (0,0,0,1), (1,0,1,0)]:
    print(init, "period =", fsr_period(init, fsr_step))

(1, 1, 0, 0) period = 5
(0, 0, 0, 0) period = 1
(0, 0, 0, 1) period = 5
(1, 0, 1, 0) period = 5


## 9. Why Plain LFSRs Are Not Enough

Linear feedback shift registers can have useful periods and efficient hardware implementations, but linearity is dangerous.

If output bits are linearly related to internal state bits, an attacker can often recover the state using linear algebra. Once the state is known, previous and future keystream bits can be reconstructed.

Modern designs add nonlinearity or use different constructions entirely.

## 10. RC4: Historically Important, No Longer Acceptable

RC4 was simple and widely deployed, including in WEP and older TLS configurations.

Its major lessons:

- Simple designs can hide subtle statistical biases.
- Nonce handling must be part of the design, not bolted on later.
- Known weaknesses should not be ignored just because exploitation seems difficult.
- Widely deployed cryptography can remain insecure for years.

RC4 should not be used in modern systems.

## 11. Modern Library Use: ChaCha20

Salsa20 and ChaCha20 are modern software-oriented stream ciphers. ChaCha20 is widely used in real systems, often paired with Poly1305 for authentication.

The `cryptography` library exposes ChaCha20 as a low-level cipher. For most real applications, you would prefer an authenticated construction such as ChaCha20-Poly1305.

The next cell attempts a ChaCha20 demonstration. If your local environment does not have `cryptography` installed, install the course requirements first.

In [9]:
try:
    from cryptography.hazmat.primitives.ciphers import Cipher, algorithms
    from cryptography.hazmat.backends import default_backend

    key = secrets.token_bytes(32)       # 256-bit key
    nonce = secrets.token_bytes(16)     # cryptography's ChaCha20 expects 128-bit nonce/counter value
    plaintext = b"ChaCha20 is a modern stream cipher design."

    algorithm = algorithms.ChaCha20(key, nonce)
    cipher = Cipher(algorithm, mode=None, backend=default_backend())
    encryptor = cipher.encryptor()
    ciphertext = encryptor.update(plaintext)

    decryptor = Cipher(algorithms.ChaCha20(key, nonce), mode=None, backend=default_backend()).decryptor()
    recovered = decryptor.update(ciphertext)

    print("ciphertext:", ciphertext.hex())
    print("recovered :", recovered)
    assert recovered == plaintext

except ImportError:
    print("cryptography is not installed. Install course requirements and rerun this cell.")

ciphertext: 5b7fc275ae917a9a2091efbdcabddc3899d059c4d96d355691c2419546745f6a7cd65b6e8157629dd5af
recovered : b'ChaCha20 is a modern stream cipher design.'


## 12. Safer Practice: Authenticated Encryption

Encryption alone only provides confidentiality. It does not prove that a ciphertext was not modified.

Modern deployments usually use authenticated encryption. For ChaCha20, a common choice is ChaCha20-Poly1305.

We will return to authentication later, but this example previews the professional pattern: use a high-level, authenticated library construction whenever possible.

In [10]:
try:
    from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305

    key = ChaCha20Poly1305.generate_key()
    aead = ChaCha20Poly1305(key)
    nonce = secrets.token_bytes(12)  # 96-bit nonce for ChaCha20-Poly1305
    aad = b"course=CYBR3570"
    plaintext = b"Authenticated encryption protects confidentiality and integrity."

    ciphertext = aead.encrypt(nonce, plaintext, aad)
    recovered = aead.decrypt(nonce, ciphertext, aad)

    print("ciphertext+tag:", ciphertext.hex())
    print("recovered     :", recovered)
    assert recovered == plaintext

except ImportError:
    print("cryptography is not installed. Install course requirements and rerun this cell.")

ciphertext+tag: 9e06b6a3d7a8a8dd7a98f491954862ddbac7d7ecce944c885acb273a41e38c6ab2f91e353c612b9d4031ebab2f2052a609e95d0166e45f05a8b2a1ccbeb794fd141885b6435590925310e26960700e15
recovered     : b'Authenticated encryption protects confidentiality and integrity.'


## 13. Tampering Check

Authenticated encryption should detect modification.

In [11]:
try:
    tampered = bytearray(ciphertext)
    tampered[0] ^= 1

    try:
        aead.decrypt(nonce, bytes(tampered), aad)
        print("Unexpected: tampered ciphertext decrypted")
    except Exception as exc:
        print("Tampering detected:", type(exc).__name__)
except NameError:
    print("Run the ChaCha20-Poly1305 cell first.")

Tampering detected: InvalidTag


## 14. Toolkit Integration

Add or update a module such as:

`crypto_toolkit/symmetric/stream.py`

Suggested functions:

- `xor_bytes(a: bytes, b: bytes) -> bytes`
- `toy_keystream(key: bytes, nonce: bytes, length: int) -> bytes`
- `toy_stream_encrypt(key: bytes, nonce: bytes, plaintext: bytes) -> bytes`
- `detect_keystream_reuse(c1: bytes, c2: bytes) -> bytes`

Remember to clearly mark toy code as educational only.

In [12]:
# Example function for toolkit integration

def detect_keystream_reuse_xor(c1: bytes, c2: bytes) -> bytes:
    """
    Return c1 XOR c2.

    If c1 and c2 were encrypted with the same stream cipher keystream,
    this equals p1 XOR p2 and may leak information about both plaintexts.
    """
    return xor_bytes(c1, c2)

## 15. Reflection

Answer the following in 5-8 sentences:

1. Why are stream ciphers useful?
2. Why is nonce reuse so dangerous?
3. Why is RC4 a good historical warning?
4. What should a developer use instead of implementing a custom stream cipher?

## Submission Checklist

- [ ] I completed the XOR exercises.
- [ ] I demonstrated key/nonce reuse failure.
- [ ] I implemented or inspected the FSR example.
- [ ] I ran the ChaCha20 or ChaCha20-Poly1305 example if my environment supports it.
- [ ] I updated my toolkit or wrote a plan for updating it.
- [ ] I completed the reflection.

REFLECTION RESPONSE!!

Stream ciphers are particularly useful because they encrypt data byte-by-byte with high speed and low latency, making them ideal for continuous network streams or resource-constrained hardware without needing padding. However, nonce reuse is catastrophic because generating the same keystream twice creates a "two-time pad" vulnerability, allowing an attacker to XOR two ciphertexts together, strip out the keystream completely, and recover the underlying plaintexts. RC4 serves as a strong historical lesson on how subtle statistical biases and poor nonce integration can degrade a widely deployed cipher over time until it becomes completely insecure. Instead of rolling custom stream ciphers, developers should rely on standardized, modern authenticated encryption constructions like ChaCha20-Poly1305 or AES-GCM provided by well-vetted cryptographic libraries. These high-level primitives protect both the confidentiality and integrity of data, ensuring tampered messages are immediately rejected.